Setup:

In [1]:
import gc
import torch

torch.cuda.empty_cache()
gc.collect()

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B', dtype=torch.bfloat16, device_map="auto")
model.seqlen = 2048

/home/mrajnoha/double-block-sparse/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:16<00:00, 17.97it/s]


In [3]:
model.hf_device_map

{'model.embed_tokens': 0,
 'model.layers.0': 0,
 'model.layers.1': 0,
 'model.layers.2': 0,
 'model.layers.3': 0,
 'model.layers.4': 0,
 'model.layers.5': 0,
 'model.layers.6': 0,
 'model.layers.7': 0,
 'model.layers.8': 0,
 'model.layers.9': 0,
 'model.layers.10': 0,
 'model.layers.11': 0,
 'model.layers.12': 0,
 'model.layers.13': 0,
 'model.layers.14': 1,
 'model.layers.15': 1,
 'model.layers.16': 1,
 'model.layers.17': 1,
 'model.layers.18': 1,
 'model.layers.19': 1,
 'model.layers.20': 1,
 'model.layers.21': 1,
 'model.layers.22': 1,
 'model.layers.23': 1,
 'model.layers.24': 1,
 'model.layers.25': 1,
 'model.layers.26': 1,
 'model.layers.27': 1,
 'model.layers.28': 1,
 'model.layers.29': 1,
 'model.layers.30': 1,
 'model.layers.31': 1,
 'model.norm': 1,
 'model.rotary_emb': 1,
 'lm_head': 1}

Tests:

In [ ]:
from llm import *

filepath_original = "./../results/original/"
filepath_pruned = "./../results/pruned/"

def calibrate(model):
    model.save_pretrained(filepath_original)

    model.eval()
    print("Retrieving C4...")
    dataloader, _ = get_c4(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)
    print("C4 retrieved.")

    tick = time.time()
    print("Running calibration...")
    model = llama_sequential(model, dataloader)
    tick_after = time.time() - tick
    minutes = tick_after // 60
    seconds = tick_after % 60
    print(f"Calibration finished in {minutes} min {seconds} s, saving model...")

    model.save_pretrained(filepath_pruned)
    print("Model saved")
    # _, testloader = get_wikitext2(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)
    # llama_eval(model, testloader)
    # print("Evaluation finished")

calibrate(model)


Writing model shards: 100%|██████████| 1/1 [00:34<00:00, 34.54s/it]


Retrieving C4...
C4 retrieved.
Running calibration...
Starting calibration...
0 self_attn.q_proj
Pruning ...
0 self_attn.k_proj
Pruning ...
0 self_attn.v_proj
Pruning ...
0 self_attn.o_proj
Pruning ...
0 mlp.gate_proj
Pruning ...
0 mlp.up_proj
Pruning ...
0 mlp.down_proj
Pruning ...
1 self_attn.q_proj
Pruning ...
1 self_attn.k_proj
Pruning ...
1 self_attn.v_proj
Pruning ...
1 self_attn.o_proj
Pruning ...
1 mlp.gate_proj
Pruning ...
1 mlp.up_proj
Pruning ...
1 mlp.down_proj
Pruning ...
2 self_attn.q_proj
Pruning ...
2 self_attn.k_proj
Pruning ...
2 self_attn.v_proj
Pruning ...
2 self_attn.o_proj
Pruning ...
2 mlp.gate_proj
Pruning ...
2 mlp.up_proj
Pruning ...
2 mlp.down_proj
Pruning ...
3 self_attn.q_proj
Pruning ...
3 self_attn.k_proj
Pruning ...
3 self_attn.v_proj
Pruning ...
3 self_attn.o_proj
Pruning ...
3 mlp.gate_proj
Pruning ...
3 mlp.up_proj
Pruning ...
3 mlp.down_proj
Pruning ...
4 self_attn.q_proj
Pruning ...
time 20.87
tensor(90624., device='cuda:0', dtype=torch.bfloat16)
4 

Writing model shards: 100%|██████████| 1/1 [00:33<00:00, 33.55s/it]

Model saved


In [ ]:
SEQLEN   = 2048
NSAMPLES = 128

_, testloader = get_wikitext2(NSAMPLES, seed=42, seqlen=SEQLEN, tokenizer=tokenizer)

ppl_dense, ppl_pruned, kl = ppl_kl_pipeline(
    filepath_dense  = filepath_original,
    filepath_pruned = filepath_pruned,
    testenc         = testloader,
    seqlen          = SEQLEN,
)

print(f"ppl_dense: {ppl_dense}")
print(f"ppl_pruned: {ppl_pruned}")
print(f"kl_results: {kl}")

### this is for PPL only
# model = AutoModelForCausalLM.from_pretrained(filepath_pruned)
# model.seqlen = SEQLEN
# ppl = llama_eval(model, testloader)
# print(f"ppl_pruned: {ppl}")
# del model

=== Layer loop: dense model ===


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2188.63it/s]


  layer 0
  layer 1
  layer 2
  layer 3
  layer 4
  layer 5
  layer 6
  layer 7
  layer 8
  layer 9
  layer 10
  layer 11
  layer 12
  layer 13
  layer 14
  layer 15
  layer 16
  layer 17
  layer 18
  layer 19
  layer 20
  layer 21
  layer 22
  layer 23
  layer 24
  layer 25
  layer 26
  layer 27
  layer 28
  layer 29
  layer 30
  layer 31
=== Layer loop: pruned model ===


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2739.65it/s]


  layer 0
  layer 1
  layer 2
  layer 3
  layer 4
  layer 5
  layer 6
  layer 7
  layer 8
  layer 9
  layer 10
  layer 11
  layer 12
  layer 13
  layer 14
  layer 15
  layer 16
  layer 17
  layer 18
  layer 19
  layer 20
  layer 21
  layer 22
  layer 23
  layer 24
  layer 25
  layer 26
  layer 27
  layer 28
  layer 29
  layer 30
  layer 31
=== Computing PPL and KL ===
PPL dense:  6.137
PPL pruned: 6.497
KL(dense || pruned)  mean=0.0579  median=0.0221  max=11.7500
ppl_dense: 6.137293815612793
ppl_pruned: 6.497016429901123
kl_results: {'mean_kl': 0.057895731180906296, 'median_kl': 0.0220947265625, 'max_kl': 11.75, 'per_token': tensor([2.9144e-03, 2.5879e-02, 4.5117e-01,  ..., 1.5503e-02, 4.1771e-04,
        5.0306e-05])}


Cleanup:

In [8]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()